In [1]:
!pip install -q apted lxml openai llama-parse

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.6/40.6 kB 1.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.1/77.1 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 362.0/362.0 kB 11.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.9/11.9 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.2/121.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 36.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 51.0/51.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.4/142.4 kB 11.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
ipython 7.34.0 requires jedi>=0.16, which is not installed.


In [2]:
import os, sys, time, json, re, base64
from pathlib import Path
import pandas as pd
from tqdm.notebook import tqdm
from lxml import html as lhtml
from apted import APTED, Config

In [3]:
from google.colab import drive
drive.mount('/content/drive')

BASE        = Path('/content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8')
SPRINT2_OUT = BASE / 'outputs' / 'sprint2'
OUT_RESULTS = SPRINT2_OUT / 'results'
OUT_GPT     = SPRINT2_OUT / 'model_outputs' / 'gpt4o_mini'
OUT_LLAMA   = SPRINT2_OUT / 'model_outputs' / 'llama'

# Separate file — won't touch Florence's benchmark_checkpoint.csv
CHECKPOINT_PATH = OUT_RESULTS / 'gpt_llama_test_benchmark.csv'

for d in [OUT_RESULTS, OUT_GPT, OUT_LLAMA]:
    d.mkdir(parents=True, exist_ok=True)

print(f'CHECKPOINT_PATH : {CHECKPOINT_PATH}')
print(f'Paths ready.')

Mounted at /content/drive
CHECKPOINT_PATH : /content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8/outputs/sprint2/results/gpt_llama_test_benchmark.csv
Paths ready.


In [4]:
import subprocess
from pathlib import Path

ZIP_PATH   = BASE / 'data' / 'hierarchical_tables_v1.zip'
EXTRACT_TO = Path('/content/data/processed')
EXTRACT_TO.mkdir(parents=True, exist_ok=True)

existing = list(EXTRACT_TO.rglob('*.png')) + list(EXTRACT_TO.rglob('*.jpg'))
if existing:
    print(f'Already extracted — {len(existing)} images.')
else:
    print('Unzipping all images...')
    subprocess.run(['unzip', '-q', str(ZIP_PATH), '-d', str(EXTRACT_TO)], capture_output=True)
    imgs = list(EXTRACT_TO.rglob('*.png')) + list(EXTRACT_TO.rglob('*.jpg'))
    print(f'Done. {len(imgs)} images extracted.')

Unzipping all images...
Done. 32670 images extracted.


In [5]:
# Build image lookup by scanning disk — never hardcode paths
IMG_LOOKUP = {}
for p in EXTRACT_TO.rglob('*.png'): IMG_LOOKUP[p.stem] = str(p)
for p in EXTRACT_TO.rglob('*.jpg'): IMG_LOOKUP[p.stem] = str(p)
print(f'Images on disk: {len(IMG_LOOKUP)}')

CSV_PATH  = BASE / 'data' / 'training_fintabnet_pool_splits_1300sample.csv'
df_splits = pd.read_csv(CSV_PATH)
df_splits['img_path'] = df_splits['img_id'].astype(str).map(IMG_LOOKUP)
df_splits['img_exists'] = df_splits['img_path'].apply(
    lambda p: Path(p).exists() if pd.notna(p) else False
)
df_valid = df_splits[df_splits['img_exists']].reset_index(drop=True)
print(f'Valid samples: {len(df_valid)} / {len(df_splits)}')

def to_samples(df):
    return df[['img_id','img_path','html','image_type']].to_dict('records')

test_samples = to_samples(df_valid[df_valid['phase']=='test'])
print(f'test_samples: {len(test_samples)}')
print(pd.DataFrame(test_samples)['image_type'].value_counts().to_string())

# Verify a sample path actually exists
s = test_samples[0]
print(f'\nSample check:')
print(f'  img_id   : {s["img_id"]}')
print(f'  img_path : {s["img_path"]}')
print(f'  exists   : {Path(s["img_path"]).exists()}')
print(f'  html     : {s["html"][:60]}')

Images on disk: 32670
Valid samples: 1300 / 1300
test_samples: 300
image_type
wide_table          60
tall_table          60
low_contrast        60
normal_table        60
low_quality_blur    60

Sample check:
  img_id   : fintabnet_009424
  img_path : /content/data/processed/data/processed/images/fintabnet_009424.png
  exists   : True
  html     : <table>
 <tr>
  <td>
  </td>
  <td colspan="5">
   Estimated


In [6]:
# ── SMOKE TEST CONTROL ───────────────────────────────────────────────
# Set True to run on 5 images only — verify pipeline before full run
# Set False for real benchmark (all test_samples)
SMOKE_TEST = False   # ← change this only

if SMOKE_TEST:
    import random
    random.seed(42)
    test_samples = random.sample(test_samples, min(5, len(test_samples)))
    print(f'⚠ SMOKE TEST — {len(test_samples)} images only')
    print('  Set SMOKE_TEST = False for full benchmark')
else:
    print(f'✓ FULL BENCHMARK — {len(test_samples)} images')

✓ FULL BENCHMARK — 300 images


In [7]:
class TableTree:
    def __init__(self,tag,colspan=1,rowspan=1,content='',children=None):
        self.tag=tag; self.colspan=int(colspan); self.rowspan=int(rowspan)
        self.content=(content or '').strip(); self.children=children or []

def _parse_el(el):
    return TableTree(el.tag,el.get('colspan',1),el.get('rowspan',1),
                     (el.text or '').strip(),[_parse_el(c) for c in el])

def html_to_tree(html_str):
    try:
        root=lhtml.fromstring(html_str.strip())
        table=root if root.tag=='table' else root.find('.//table')
        return _parse_el(table) if table is not None else TableTree('empty')
    except: return TableTree('error')

def _size(node): return 1+sum(_size(c) for c in node.children)

class _TEDSConfig(Config):
    def rename(self,a,b):
        if a.tag!=b.tag: return 1.0
        if a.colspan!=b.colspan or a.rowspan!=b.rowspan: return 1.0
        if a.content!=b.content: return 0.5
        return 0.0
    def children(self,n): return n.children
    def insert(self,n): return 1.0
    def delete(self,n): return 1.0

def compute_teds(pred_html, true_html):
    if not pred_html or not pred_html.strip(): return 0.0
    pt=html_to_tree(pred_html); tt=html_to_tree(true_html)
    if pt.tag in ('empty','error') or tt.tag in ('empty','error'): return 0.0
    dist=APTED(pt,tt,_TEDSConfig()).compute_edit_distance()
    denom=max(_size(pt),_size(tt))
    return max(0.0,1.0-dist/denom) if denom else 1.0

# Sanity checks
_gt  = "<table><tr><th colspan='2'>Revenue</th></tr><tr><td>Q1</td><td>Q2</td></tr></table>"
_bad = "<table><tr><th>Revenue</th><th>Revenue</th></tr><tr><td>Q1</td><td>Q2</td></tr></table>"
assert compute_teds(_gt,_gt)==1.0
assert compute_teds(_bad,_gt)<1.0
assert compute_teds('',_gt)==0.0
print('TEDS scorer ready.')

TEDS scorer ready.


In [8]:
def classify_failure(pred_html, true_html, teds_score):
    f=dict(header_collapse=False,colspan_ignored=False,rowspan_ignored=False,
           table_not_found=False,cell_count_wrong=False,content_mismatch=False)
    if not pred_html or not pred_html.strip():
        f['table_not_found']=True; return f
    try:
        pt=lhtml.fromstring(pred_html).find('.//table')
        tt=lhtml.fromstring(true_html).find('.//table')
        if pt is None: f['table_not_found']=True; return f
        p_hrows=len(pt.findall('.//thead/tr'))+len(pt.findall('.//th'))
        t_hrows=len(tt.findall('.//thead/tr'))+len(tt.findall('.//th'))
        if t_hrows>1 and p_hrows<t_hrows: f['header_collapse']=True
        def spans(el,attr): return [int(x.get(attr,1)) for x in el.findall('.//*') if x.tag in ('td','th')]
        if any(c>1 for c in spans(tt,'colspan')) and not any(c>1 for c in spans(pt,'colspan')): f['colspan_ignored']=True
        if any(r>1 for r in spans(tt,'rowspan')) and not any(r>1 for r in spans(pt,'rowspan')): f['rowspan_ignored']=True
        p_c=len([x for x in pt.findall('.//*') if x.tag in ('td','th')])
        t_c=len([x for x in tt.findall('.//*') if x.tag in ('td','th')])
        if t_c>0 and abs(p_c-t_c)/t_c>0.20: f['cell_count_wrong']=True
        if not any(f.values()) and teds_score<0.8: f['content_mismatch']=True
    except: f['table_not_found']=True
    return f

def markdown_to_html_table(md_text):
    if not md_text: return ''
    lines=md_text.splitlines(); table_lines=[]; in_table=False
    for line in lines:
        s=line.strip()
        if s.startswith('|') and s.endswith('|'): in_table=True; table_lines.append(s)
        elif in_table: break
    if not table_lines: return ''
    is_sep=lambda l: re.fullmatch(r'[|\-: ]+',l) is not None
    data_rows=[l for l in table_lines if not is_sep(l)]
    if not data_rows: return ''
    html='<table>\n'
    for i,row in enumerate(data_rows):
        cells=[c.strip() for c in row.strip('|').split('|')]
        tag='th' if i==0 else 'td'
        html+='  <tr>'+''.join(f'<{tag}>{c}</{tag}>' for c in cells)+'</tr>\n'
    return html+'</table>'

def extract_html_table(raw_text):
    if not raw_text: return ''
    match=re.search(r'(<table[\s\S]*?</table>)',raw_text,re.IGNORECASE)
    if match: return match.group(1)
    return markdown_to_html_table(raw_text)

print('Failure classifier + utilities ready.')

Failure classifier + utilities ready.


In [9]:
import openai
from google.colab import userdata
from llama_parse import LlamaParse

OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
LLAMA_API_KEY  = userdata.get('LLAMA_API_KEY')

print(f'OpenAI  : {"✓ key found" if OPENAI_API_KEY else "✗ MISSING — add OPENAI_API_KEY to Colab Secrets"}')
print(f'LlamaParse: {"✓ key found" if LLAMA_API_KEY else "✗ MISSING — add LLAMA_API_KEY to Colab Secrets"}')


def run_gpt4o_mini(img_path):
    img_path = Path(img_path); t0 = time.time()
    if not OPENAI_API_KEY: return '', 0.0
    try:
        with open(img_path, 'rb') as f: img_b64 = base64.b64encode(f.read()).decode()
        mime   = 'image/png' if img_path.suffix.lower()=='.png' else 'image/jpeg'
        client = openai.OpenAI(api_key=OPENAI_API_KEY)
        resp   = client.chat.completions.create(
            model='gpt-4o-mini',
            messages=[{'role':'user','content':[
                {'type':'image_url','image_url':{'url':f'data:{mime};base64,{img_b64}'}},
                {'type':'text','text':(
                    'Extract the table from this image as valid HTML.\n'
                    'Rules:\n'
                    '- Use <table>, <tr>, <th>, <td> tags only\n'
                    '- Preserve colspan and rowspan for ALL merged/spanning cells\n'
                    '- Use <th> for header cells, <td> for data cells\n'
                    '- Return ONLY the raw HTML starting with <table> and ending with </table>'
                )}
            ]}],
            max_tokens=2048, temperature=0,
        )
        return extract_html_table(resp.choices[0].message.content.strip()), round(time.time()-t0,2)
    except openai.AuthenticationError:
        print('  [GPT-4o-mini] Invalid API key'); return '', round(time.time()-t0,2)
    except Exception as e:
        print(f'  [GPT-4o-mini] Error on {img_path.name}: {e}'); return '', round(time.time()-t0,2)


def run_llamaparse(img_path):
    img_path = Path(img_path); t0 = time.time()
    if not LLAMA_API_KEY: return '', 0.0
    try:
        parser = LlamaParse(api_key=LLAMA_API_KEY, result_type='markdown', verbose=False)
        docs   = parser.load_data(str(img_path))
        if not docs: return '', round(time.time()-t0,2)
        # LlamaParse returns markdown — use markdown_to_html_table, not extract_html_table
        return markdown_to_html_table(docs[0].text), round(time.time()-t0,2)
    except Exception as e:
        print(f'  [LlamaParse] Error on {img_path.name}: {e}'); return '', round(time.time()-t0,2)

print('Runners ready.')

/tmp/ipykernel_90931/2230301443.py:3: DeprecationWarning: The 'llama-parse' package is deprecated and will no longer receive updates. Please migrate to the new unified SDK. See https://developers.llamaindex.ai/python/cloud/llamaparse/getting_started/ and https://github.com/run-llama/llama-cloud-py/blob/main/README.md for migration instructions.
  from llama_parse import LlamaParse


OpenAI  : ✓ key found
LlamaParse: ✓ key found
Runners ready.


In [10]:
def run_single_model(prefix, name, runner, out_dir):
    out_dir.mkdir(parents=True, exist_ok=True)
    df_ckpt  = pd.read_csv(CHECKPOINT_PATH) if CHECKPOINT_PATH.exists() else pd.DataFrame()
    teds_col = f'{prefix}_teds'
    done_ids = set(df_ckpt.loc[df_ckpt[teds_col].notna(),'image_id'].tolist()) \
               if teds_col in df_ckpt.columns else set()
    remaining = [s for s in test_samples if s['img_id'] not in done_ids]
    print(f'\n[{name}] {len(done_ids)} already done — {len(remaining)} remaining')

    for item in tqdm(remaining, desc=name):
        img_path  = Path(item['img_path'])
        true_html = item['html']
        img_id    = item['img_id']

        pred_html, elapsed = runner(img_path)
        teds  = compute_teds(pred_html, true_html)
        fails = classify_failure(pred_html, true_html, teds)

        (out_dir/f'{img_id}.html').write_text(pred_html or '', encoding='utf-8')

        new_cols = {
            'image_id'        : img_id,
            'image_type'      : item['image_type'],
            f'{prefix}_teds'  : round(teds,4),
            f'{prefix}_time_s': elapsed,
        }
        for k,v in fails.items(): new_cols[f'{prefix}_{k}'] = v

        df_ckpt = pd.read_csv(CHECKPOINT_PATH) if CHECKPOINT_PATH.exists() else pd.DataFrame()
        if 'image_id' in df_ckpt.columns and img_id in df_ckpt['image_id'].values:
            for col,val in new_cols.items():
                df_ckpt.loc[df_ckpt['image_id']==img_id, col] = val
        else:
            df_ckpt = pd.concat([df_ckpt,pd.DataFrame([new_cols])],ignore_index=True)
        df_ckpt.to_csv(CHECKPOINT_PATH, index=False)

    print(f'[{name}] Done → {CHECKPOINT_PATH}')
    return pd.read_csv(CHECKPOINT_PATH)

print('Benchmark loop ready.')

Benchmark loop ready.


In [13]:
df = run_single_model('gpt4o_mini', 'GPT-4o-mini', run_gpt4o_mini, OUT_GPT)
print(df[['image_id','image_type','gpt4o_mini_teds']].tail(5).to_string(index=False))


[GPT-4o-mini] 114 already done — 186 remaining


GPT-4o-mini:   0%|          | 0/186 [00:00<?, ?it/s]

[GPT-4o-mini] Done → /content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8/outputs/sprint2/results/gpt_llama_test_benchmark.csv
        image_id       image_type  gpt4o_mini_teds
fintabnet_009760 low_quality_blur           0.5365
fintabnet_010978 low_quality_blur           0.4808
fintabnet_018425 low_quality_blur           0.4912
fintabnet_030282 low_quality_blur           0.6557
fintabnet_016727 low_quality_blur           0.4811


In [15]:
df = run_single_model('llama', 'LlamaParse', run_llamaparse, OUT_LLAMA)
print(df[['image_id','image_type','llama_teds']].tail(5).to_string(index=False))


[LlamaParse] 241 already done — 59 remaining


LlamaParse:   0%|          | 0/59 [00:00<?, ?it/s]

[LlamaParse] Done → /content/drive/My Drive/Quarter 3/Computer Vision and DL/Computer_Vision_Team8/outputs/sprint2/results/gpt_llama_test_benchmark.csv
        image_id       image_type  llama_teds
fintabnet_009760 low_quality_blur      0.7152
fintabnet_010978 low_quality_blur      0.3206
fintabnet_018425 low_quality_blur      0.3860
fintabnet_030282 low_quality_blur      0.4410
fintabnet_016727 low_quality_blur      0.5849


In [16]:
df = pd.read_csv(CHECKPOINT_PATH)
teds_cols = [c for c in df.columns if c.endswith('_teds')]

print('Average TEDS by image type:')
print(df.groupby('image_type')[teds_cols].mean().round(4).to_string())
print('\nOverall average TEDS:')
print(df[teds_cols].mean().round(4).to_string())

Average TEDS by image type:
                  gpt4o_mini_teds  llama_teds
image_type                                   
low_contrast               0.4891      0.5277
low_quality_blur           0.5664      0.5132
normal_table               0.5375      0.4782
tall_table                 0.3416      0.4537
wide_table                 0.4963      0.5526

Overall average TEDS:
gpt4o_mini_teds    0.4862
llama_teds         0.5051
